# 03 · Ablations

One config switch each, so each row attributes something. Every delta is read against `sqrt(2) * seed_sd` — the run-to-run scale of a difference between two runs. A bar inside that band contributed nothing measurable, whatever its p-value.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
os.environ.setdefault("OMP_NUM_THREADS", "2")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch; torch.set_num_threads(2)

TABLES = os.path.join("..", "results", "tables")
def table(name):
    return pd.read_csv(os.path.join(TABLES, name))
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)


In [ ]:
summary = table('ablation_summary.csv')
cols = ['ablation', 'der', 'der_seed_sd', 'der_noise_scale', 'confusion',
        'n_speakers_pred', 'extra_mean_query_size']
print(summary[[c for c in cols if c in summary.columns]].round(4)
      .to_string(index=False))

`extra_mean_query_size` is the diagnostic that shows the bounded-window mechanism is actually engaged: it is the number of windows averaged into each decision's query. It must be 1.00 exactly when `bounded_window: false`, and above 1 otherwise. If it were 1.00 for the full system, the ablation would be testing nothing.

In [ ]:
tests = table('ablation_tests.csv')
sub = tests[tests.metric == 'der']
print(sub[['name_b', 'mean_a', 'mean_b', 'delta', 'ci_lower', 'ci_upper',
           'p_adjusted', 'noise_scale', 'noise_ratio', 'verdict']]
      .round(5).to_string(index=False))

In [ ]:
sub = sub.copy()
sub['cost'] = -sub['delta']
sub = sub.sort_values('cost')
noise = float(np.nanmax(sub['noise_scale']))
plt.figure(figsize=(9, 4))
colors = ['#4c78a8' if v in ('survives', 'suggestive') else '#bbbbbb'
          for v in sub.verdict]
plt.barh(np.arange(len(sub)), sub['cost'], color=colors)
plt.axvspan(-noise, noise, color='k', alpha=0.10,
            label=r'$\pm\sqrt{2}\sigma_{seed}$')
plt.axvline(0, color='k', lw=1)
plt.yticks(np.arange(len(sub)),
           [f'{r.name_b}  [{r.verdict}]' for r in sub.itertuples()],
           fontsize=8)
plt.xlabel('DER change when the mechanism is removed')
plt.grid(alpha=0.3, axis='x'); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

### The non-causal row is not a system

`noncausal` uses centred convolution padding, so each frame's representation depends on roughly 150 ms of *future* audio. Its reported emission latency therefore understates its true latency by that amount, and it is a diagnostic rather than a deployable configuration. `tests/test_causality.py` requires it to **fail** the bit-identity check the shipped model passes.